# Notebook 4: Feature Importance and SHAP Analysis

**Author**: Laura Steel Pascual  
**Date**: February 2026  
**Project**: Decoding Anthropogenic Impact with eDNA and Supervised Machine Learning: Species‑Level Associations from Solomon Island Rivers

---

## Overview

This notebook identifies species that are most important for predicting ecological condition using two complementary analytical approaches. Analysis covers seven statistically significant Random Forest models (p < 0.05) identified in Notebook 3, using both Gini importance metrics and SHAP (SHapley Additive exPlanations) values to reveal which species drive model predictions. Cross-validation ensures robust estimates, while cross-model consistency analysis identifies species that serve as reliable indicators across multiple spatial scales and impact thresholds.

**Purpose**:
- Calculate Gini Importance (from Random Forest): Measures how frequently and effectively each species is used for splitting decisions across all decision trees in the ensemble.
- Calculate SHAP Values (SHapley Additive exPlanations): Quantifies each species' contribution to individual site predictions while accounting for feature interactions, providing both magnitude and direction of effects.

**Scope:**
- 7 statistically significant models from Notebook 3 (p < 0.05).
- Excluded: 500m / 0.03125 threshold (p = 0.103, not significant).

**Paper Reference**: 2. Materials & Methods -> 2.7 Feature Importance and Interpretability; 3. Results -> 3.3 Feature Importance and SHAP Analyses

---

## 1. Setup

In [3]:
# Import required libraries
import numpy as np
import pandas as pd
import sklearn
import shap
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
sns.set_style('whitegrid')
import warnings
warnings.filterwarnings('ignore')
import os
import pickle
import copy  

# Create output directories
os.makedirs('results/feature_importance', exist_ok=True)
os.makedirs('results/shap_analysis', exist_ok=True)
os.makedirs('results/shap_analysis/figures', exist_ok=True)
os.makedirs('results/common_species', exist_ok=True)

# Verify package versions
print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')
print(f'Scikit-learn version: {sklearn.__version__}')
print(f'SHAP version: {shap.__version__}')
print(f'Seaborn version: {sns.__version__}')

NumPy version: 1.24.4
Pandas version: 2.3.2
Scikit-learn version: 1.2.2
SHAP version: 0.48.0
Seaborn version: 0.12.2


## 2. Configuration

Define the statistically significant models to analyse (only those with p < 0.05).

In [5]:
# Significant models from Notebook 3 (p < 0.05)
# Note: 500m_t003125 excluded (p = 0.103, not significant)
significant_models = {
    '100m_t003125': {'buffer': 100, 'threshold': 0.03125, 'p_value': 0.043, 'sig': '*'},
    '100m_t00625': {'buffer': 100, 'threshold': 0.0625, 'p_value': 0.024, 'sig': '*'},
    '300m_t003125': {'buffer': 300, 'threshold': 0.03125, 'p_value': 0.004, 'sig': '**'},
    '300m_t00625': {'buffer': 300, 'threshold': 0.0625, 'p_value': 0.010, 'sig': '*'},
    '500m_t00625': {'buffer': 500, 'threshold': 0.0625, 'p_value': 0.005, 'sig': '*'},
    '1000m_t003125': {'buffer': 1000, 'threshold': 0.03125, 'p_value': 0.000, 'sig': '***'},
    '1000m_t00625': {'buffer': 1000, 'threshold': 0.0625, 'p_value': 0.000, 'sig': '***'},
}

# Cross-validation parameters (matches Notebook 2 training)
n_folds = 5
random_state = 42
cv_strategy = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)

print(f'\nAnalysing {len(significant_models)} statistically significant models (p < 0.05)')
print('Excluded: 500m_t003125 (p = 0.103, not significant)\n')


Analysing 7 statistically significant models (p < 0.05)
Excluded: 500m_t003125 (p = 0.103, not significant)



## 3. Load Models and Data

Load the trained Random Forest pipelines and preprocessed datasets from Notebook 2. Each model consists of a pipeline with three steps: log transformation -> standard scaling -> Random Forest classifier.

In [7]:
models = {}
datasets = {}

for key, info in significant_models.items():
    # Load preprocessed data (X_raw, y labels, site IDs)
    data_path = f'results/preprocessed/data_{key}.pkl'
    with open(data_path, 'rb') as f:
        datasets[key] = pickle.load(f)
    
    # Load trained model pipeline 
    # These models were trained on ALL 47 samples in Notebook 2
    model_path = f'results/models/rf_optimised_{key}.pkl'
    with open(model_path, 'rb') as f:
        models[key] = pickle.load(f)
    
    n_samples = len(datasets[key]['y'])
    n_features = datasets[key]['X_raw'].shape[1]
    
    print(f'{key}: {n_samples} samples, {n_features} features (p = {info["p_value"]:.3f} {info["sig"]})')

print(f'\nLoaded {len(models)} models successfully')

100m_t003125: 47 samples, 861 features (p = 0.043 *)
100m_t00625: 47 samples, 861 features (p = 0.024 *)
300m_t003125: 47 samples, 861 features (p = 0.004 **)
300m_t00625: 47 samples, 861 features (p = 0.010 *)
500m_t00625: 47 samples, 861 features (p = 0.005 *)
1000m_t003125: 47 samples, 861 features (p = 0.000 ***)
1000m_t00625: 47 samples, 861 features (p = 0.000 ***)

Loaded 7 models successfully


## 4. Extract Feature Names

Retrieve species names. 

**Reminder**: All models use the same 861 species features; only the target variable (impact classification) differs between configurations.

In [9]:
sample_key = list(datasets.keys())[0]
feature_names = datasets[sample_key]['feature_names']
n_total_features = len(feature_names)

print(f'Total features: {n_total_features}')
print(f'Sample features: {feature_names[:5]}...')

Total features: 861
Sample features: ['Sicyopterus cynocephalus or lagocephalus', 'Areca catechu', 'Rhyacichthys guilberti', 'Stiphodon semoni', 'Kleinhovia hospita']...


## 5. Calculate Feature Importances and SHAP Values

For each model, calculate two importance metrics using 5-fold cross-validation to ensure robust estimates.

**Process**:

For each CV fold:
- Train model on ~38 training samples.
- Extract Gini importance from the trained Random Forest.
- Calculate SHAP values on training samples.

Aggregate across folds:
- Compute mean and standard deviation for both metrics.
- Create ranked species lists.

For each model, mean ± standard deviation of Gini importance and SHAP values for all 861 species.

In [11]:
print('\n' + '='*80)
print('Feature Importance and SHAP Values')
print('='*80)

all_importances = {}
all_shap_values = {}

for key, info in significant_models.items():
    print(f'\n{key} (p = {info["p_value"]:.3f} {info["sig"]})')
    print('-'*80)
    
    # Get data and ORIGINAL pipeline (trained on all 47 samples)
    X = datasets[key]['X_raw']
    y = datasets[key]['y']
    pipe_original = models[key]  # Keep this safe - do NOT modify!
    
    # Storage for fold results
    importances_matrix = []
    shap_matrix = []
    X_combined = []  # Store transformed features from all folds
    
    # Process each fold
    for fold_idx, (train_idx, _) in enumerate(cv_strategy.split(X, y)):
        print(f'  Fold {fold_idx + 1}/{n_folds}...', end=' ')
        
        # Get training data
        X_train = X[train_idx]
        y_train = y[train_idx]
        
        # CRITICAL FIX: Create a COPY to avoid overwriting the original model
        pipe = copy.deepcopy(pipe_original)
        
        # Fit the COPY (not the original!)
        pipe.fit(X_train, y_train)
        
        # Extract feature importances from Random Forest
        rf_model = pipe.named_steps['clf']
        importances_matrix.append(rf_model.feature_importances_)
        
        # Transform training data for SHAP
        X_train_transformed = pipe.named_steps['scaler'].transform(
            pipe.named_steps['log_transform'].transform(X_train)
        )
        X_train_df = pd.DataFrame(
            X_train_transformed, 
            columns=feature_names,
            index=range(len(X_train))
        )
        
        # Store transformed features
        X_combined.append(X_train_df)
        
        # Calculate SHAP values
        explainer = shap.TreeExplainer(rf_model)
        shap_values = explainer.shap_values(X_train_df)
        
        # Extract class 1 (impacted) SHAP values
        if isinstance(shap_values, list) or (isinstance(shap_values, np.ndarray) and shap_values.ndim == 3):
            shap_class_1 = shap_values[:, :, 1] if shap_values.ndim == 3 else shap_values[1]
        else:
            shap_class_1 = shap_values
        
        # Store absolute SHAP values
        shap_matrix.append(np.abs(shap_class_1))
        
        print('Done')
    
    # Calculate mean and std across folds
    importances_array = np.array(importances_matrix)
    mean_importance = np.mean(importances_array, axis=0)
    std_importance = np.std(importances_array, axis=0)
    
    shap_full_matrix = np.vstack(shap_matrix)
    X_full_df = pd.concat(X_combined, ignore_index=True)  # Combine all folds
    mean_shap = np.mean(shap_full_matrix, axis=0)
    std_shap = np.std(shap_full_matrix, axis=0)
    
    # Store results
    all_importances[key] = {
        'mean': mean_importance,
        'std': std_importance,
        'matrix': importances_array
    }
    
    all_shap_values[key] = {
        'mean': mean_shap,
        'std': std_shap,
        'matrix': shap_full_matrix,
        'features': X_full_df  
    }
    
    # Display top 5
    imp_df_temp = pd.DataFrame({
        'Species': feature_names,
        'Mean Importance': mean_importance
    }).sort_values(by='Mean Importance', ascending=False)
    
    shap_df_temp = pd.DataFrame({
        'Species': feature_names,
        'Mean SHAP': mean_shap
    }).sort_values(by='Mean SHAP', ascending=False)
    
    print(f'\n  Top 5 by Importance: {", ".join(imp_df_temp.head(5)["Species"].values)}')
    print(f'  Top 5 by SHAP:       {", ".join(shap_df_temp.head(5)["Species"].values)}')

print('\n' + '='*80)
print('Feature importance and SHAP calculations complete')
print('='*80)

# VERIFICATION: Confirm original models are still intact
print('\nVerifying original models preserved:')
sample_key = list(models.keys())[0]
n_samples_seen = models[sample_key].named_steps['scaler'].n_samples_seen_
print(f'  {sample_key} scaler n_samples_seen_: {n_samples_seen}')
if n_samples_seen == 47:
    print('  Correct: Models trained on all 47 samples')
else:
    print(f'  ERROR: Models show {n_samples_seen} samples (should be 47)')


Feature Importance and SHAP Values

100m_t003125 (p = 0.043 *)
--------------------------------------------------------------------------------
  Fold 1/5... Done
  Fold 2/5... Done
  Fold 3/5... Done
  Fold 4/5... Done
  Fold 5/5... Done

  Top 5 by Importance: Kleinhovia hospita, Canis lupus familiaris, Sicyopterus cynocephalus or lagocephalus, Chilomonas sp. M1303, uncultured eukaryote
  Top 5 by SHAP:       Kleinhovia hospita, Canis lupus familiaris, uncultured eukaryote, Chilomonas sp. M1303, Sicyopterus cynocephalus or lagocephalus

100m_t00625 (p = 0.024 *)
--------------------------------------------------------------------------------
  Fold 1/5... Done
  Fold 2/5... Done
  Fold 3/5... Done
  Fold 4/5... Done
  Fold 5/5... Done

  Top 5 by Importance: Canis lupus familiaris, Kleinhovia hospita, Chilomonas sp. M1303, Crenimugil cf. heterocheilos, Sicyopterus stiphodonoides
  Top 5 by SHAP:       Canis lupus familiaris, Kleinhovia hospita, Piper methysticum, Sicyopterus stiphod

## 6. Save Results

In [13]:
for key in significant_models.keys():
    # Feature Importance DataFrame
    imp_df = pd.DataFrame({
        'Species': feature_names,
        'Mean Importance': all_importances[key]['mean'],
        'Std Importance': all_importances[key]['std']
    }).sort_values(by='Mean Importance', ascending=False)
    
    imp_path = f'results/feature_importance/importance_{key}.xlsx'
    imp_df.to_excel(imp_path, index=False, sheet_name='Feature Importance')
    
    # SHAP Values DataFrame
    shap_df = pd.DataFrame({
        'Species': feature_names,
        'Mean SHAP': all_shap_values[key]['mean'],
        'Std SHAP': all_shap_values[key]['std']
    }).sort_values(by='Mean SHAP', ascending=False)
    
    shap_path = f'results/shap_analysis/shap_{key}.xlsx'
    shap_df.to_excel(shap_path, index=False, sheet_name='SHAP Values')
    
    print(f'  Saved: {key}')

print('\nAll Excel files saved')

  Saved: 100m_t003125
  Saved: 100m_t00625
  Saved: 300m_t003125
  Saved: 300m_t00625
  Saved: 500m_t00625
  Saved: 1000m_t003125
  Saved: 1000m_t00625

All Excel files saved


## 7. Visualisations

Generate two types of visualisations for each model:
- SHAP Summary Plot (Beeswarm): Shows the distribution of SHAP values for the top 20 features. Each point represents one sample; color indicates feature value (pink = high, blue = low).
- Importance Comparison Bar Chart: Compares the top 20 species by SHAP value with their corresponding Gini importance scores.

In [15]:
print('\n' + '='*80)
print('Visualisations')
print('='*80)

def plot_shap_summary(shap_matrix, X_features, key, info, save_path):
    """Create SHAP summary plot (beeswarm) for top 20 features"""
    
    # Get top 20 features by mean absolute SHAP
    mean_shap = np.mean(np.abs(shap_matrix), axis=0)
    top_20_idx = np.argsort(mean_shap)[::-1][:20]
    
    # Subset SHAP values and features for top 20
    shap_subset = shap_matrix[:, top_20_idx]
    X_subset = X_features.iloc[:, top_20_idx]
    
    plt.figure(figsize=(6, 14))
    shap.summary_plot(shap_subset, X_subset, show=False)
    
    ax = plt.gca()
    ax.tick_params(axis='both', labelsize=10)
    ax.set_xlabel('SHAP value', fontsize=10)
    ax.set_title(f'Top 20 SHAP Summary Plot\n', fontsize=12, weight='bold')
    ax.text(0.33, 1.01, f'RFChp_{key}_5f', transform=ax.transAxes, 
            fontsize=8, color='gray')
    ax.text(0.33, 0.99, f'p = {info["p_value"]:.3f} {info["sig"]}', 
            transform=ax.transAxes, fontsize=8, color='gray')
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_path, f'shap_summary_{key}.png'), dpi=300, bbox_inches='tight')
    plt.close()

def plot_importance_comparison(imp_mean, imp_std, shap_mean, shap_std, 
                               feature_names, key, info, save_path):
    """Create bar chart comparing top 20 SHAP features with their Gini importance"""
    
    # Get top 20 by SHAP
    top_20_idx = np.argsort(shap_mean)[::-1][:20]
    
    species = [feature_names[i] for i in top_20_idx]
    shap_vals = shap_mean[top_20_idx]
    shap_errs = shap_std[top_20_idx]
    imp_vals = imp_mean[top_20_idx]
    imp_errs = imp_std[top_20_idx]
    
    # Plot
    plt.style.use('default')
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(species))
    width = 0.4
    
    ax.bar(x - width/2, imp_vals, width, yerr=imp_errs, 
           label='Model Importance', color='lightsteelblue', 
           edgecolor='black', capsize=3)
    ax.bar(x + width/2, shap_vals, width, yerr=shap_errs, 
           label='SHAP Value', color='slategray', 
           edgecolor='black', capsize=3)
    
    ax.set_xticks(x)
    ax.set_xticklabels(species, rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Mean Value', fontsize=10)
    ax.set_title('Top 20 Features by SHAP Value (with Model Importance)\n', 
                 fontsize=12, weight='bold')
    ax.text(0.01, 1.05, f'RFChp_{key}_5f', transform=ax.transAxes, 
            fontsize=8, color='gray')
    ax.text(0.01, 1.02, f'p = {info["p_value"]:.3f} {info["sig"]}', 
            transform=ax.transAxes, fontsize=8, color='gray')
    ax.legend(frameon=False, fontsize=9)
    ax.grid(True, linestyle='--', linewidth=0.5, alpha=0.3)
    ax.set_facecolor('white')
    fig.tight_layout()
    fig.savefig(os.path.join(save_path, f'importance_comparison_{key}.png'), dpi=300)
    plt.close()

# Generate plots for each model
for key, info in significant_models.items():
    print(f'\nGenerating plots for {key}...')
    
    # SHAP summary plot (uses full matrix + actual feature values)
    plot_shap_summary(
        all_shap_values[key]['matrix'],
        all_shap_values[key]['features'],
        key,
        info,
        'results/shap_analysis/figures'
    )
    print(f'  SHAP summary plot')
    
    # Importance comparison (uses aggregated mean/std)
    plot_importance_comparison(
        all_importances[key]['mean'],
        all_importances[key]['std'],
        all_shap_values[key]['mean'],
        all_shap_values[key]['std'],
        feature_names,
        key,
        info,
        'results/shap_analysis/figures'
    )
    print(f'  Importance comparison plot')

print('\n' + '='*80)
print('All visualisations complete')
print('='*80)


Visualisations

Generating plots for 100m_t003125...
  SHAP summary plot
  Importance comparison plot

Generating plots for 100m_t00625...
  SHAP summary plot
  Importance comparison plot

Generating plots for 300m_t003125...
  SHAP summary plot
  Importance comparison plot

Generating plots for 300m_t00625...
  SHAP summary plot
  Importance comparison plot

Generating plots for 500m_t00625...
  SHAP summary plot
  Importance comparison plot

Generating plots for 1000m_t003125...
  SHAP summary plot
  Importance comparison plot

Generating plots for 1000m_t00625...
  SHAP summary plot
  Importance comparison plot

All visualisations complete


## 8. Cross-Model Analysis

Identify species that consistently appear in the top rankings across multiple models. This reveals which species are robust indicators regardless of spatial scale or impact threshold.

Model Groups Analysed:
- All significant models (p < 0.05): 7 models.
- Strong significance (p < 0.005): 4 models.
- Very strong significance (p < 0.001), which is also the buffer-specific group 1000m: 2 models.
- Buffer-specific groups (100m, 300-500m).
- Threshold tendency groups (leaning toward reference vs impacted).

**Key Finding**: 61 unique species identified as important indicators across at least one model group.

In [17]:
print('\n' + '='*80)
print('Cross-model analysis')
print('='*80)

def find_common_species(model_keys, top_n, metric='shap'):
    """Find species appearing in top N for all specified models"""
    species_sets = []
    
    for key in model_keys:
        if metric == 'shap':
            df = pd.read_excel(f'results/shap_analysis/shap_{key}.xlsx')
            species_sets.append(set(df.head(top_n)['Species'].values))
        else:  # importance
            df = pd.read_excel(f'results/feature_importance/importance_{key}.xlsx')
            species_sets.append(set(df.head(top_n)['Species'].values))
    
    return set.intersection(*species_sets)

# Define model groups
model_groups = {
    'sig0.05': {
        'models': ['100m_t003125', '100m_t00625', '300m_t003125', '300m_t00625', 
                   '500m_t00625', '1000m_t003125', '1000m_t00625'],
        'description': 'p < 0.05'
    },
    'sig0.005': {
        'models': ['300m_t003125', '500m_t00625', '1000m_t003125', '1000m_t00625'],
        'description': 'p < 0.005'
    },
    'sig0.005_lean1': {
        'models': ['300m_t003125', '1000m_t003125'],
        'description': 'p < 0.005, leaning towards impacted'
    },
    'sig0.005_lean0': {
        'models': ['500m_t00625', '1000m_t00625'],
        'description': 'p < 0.005, leaning towards reference'
    },
    'sig0.001_1000m': {
        'models': ['1000m_t003125', '1000m_t00625'],
        'description': 'p < 0.001'
    },
    '100m': {
        'models': ['100m_t003125', '100m_t00625'],
        'description': 'p < 0.05, 100m'
    },
    '300500m': {
        'models': ['300m_t003125', '300m_t00625', '500m_t00625'],
        'description': 'p < 0.05, 300-500m'
    }
}

print('\nModel groups:')
for group_name, group_info in model_groups.items():
    print(f'  {group_name}: {len(group_info["models"])} models - {group_info["description"]}')

# Create Excel writer for consolidated output
writer = pd.ExcelWriter('results/common_species/common_species_summary.xlsx', engine='openpyxl')

# Analyse common species for each group
for group_name, group_info in model_groups.items():
    model_list = group_info['models']
    description = group_info['description']
    
    print(f'\n{group_name} ({description})')
    print('-'*80)
    
    # Storage for this group's results
    group_results = {'Top_N': [], 'SHAP': [], 'Importance': []}
    
    for top_n in [10, 20, 30, 43]:
        # SHAP common species
        shap_common = find_common_species(model_list, top_n, 'shap')
        
        # Importance common species
        imp_common = find_common_species(model_list, top_n, 'importance')
        
        print(f'  Top {top_n:2d}: SHAP={len(shap_common):2d} species, Importance={len(imp_common):2d} species')
        
        # Store for consolidated sheet
        group_results['Top_N'].append(top_n)
        group_results['SHAP'].append(', '.join(sorted(shap_common)) if shap_common else 'None')
        group_results['Importance'].append(', '.join(sorted(imp_common)) if imp_common else 'None')
    
    # Save consolidated results for this group to one sheet
    group_df = pd.DataFrame(group_results)
    group_df.to_excel(writer, sheet_name=group_name, index=False)

# Save and close writer
writer.close()

print('\n' + '='*80)
print('Cross-model analysis complete')
print('Saved: results/common_species/common_species_summary.xlsx')
print('='*80)


Cross-model analysis

Model groups:
  sig0.05: 7 models - p < 0.05
  sig0.005: 4 models - p < 0.005
  sig0.005_lean1: 2 models - p < 0.005, leaning towards impacted
  sig0.005_lean0: 2 models - p < 0.005, leaning towards reference
  sig0.001_1000m: 2 models - p < 0.001
  100m: 2 models - p < 0.05, 100m
  300500m: 3 models - p < 0.05, 300-500m

sig0.05 (p < 0.05)
--------------------------------------------------------------------------------
  Top 10: SHAP= 0 species, Importance= 0 species
  Top 20: SHAP= 1 species, Importance= 1 species
  Top 30: SHAP= 1 species, Importance= 2 species
  Top 43: SHAP= 3 species, Importance= 4 species

sig0.005 (p < 0.005)
--------------------------------------------------------------------------------
  Top 10: SHAP= 0 species, Importance= 0 species
  Top 20: SHAP= 2 species, Importance= 2 species
  Top 30: SHAP= 3 species, Importance= 3 species
  Top 43: SHAP= 6 species, Importance= 7 species

sig0.005_lean1 (p < 0.005, leaning towards impacted)
----

## 9. Key Species Identification

Collect all common species from top 43 (5%) across all groups, but only use the unique SHAP 5% to for dependence plots.

In [19]:
print('\n' + '='*80)
print('Key species identification')
print('='*80)

all_key_species_shap = set()
all_key_species_importance = set()

print('\nCollecting top 43 common species from each group:')
print('-'*80)

for group_name, group_info in model_groups.items():
    model_list = group_info['models']
    
    shap_common_43 = find_common_species(model_list, 43, 'shap')
    imp_common_43 = find_common_species(model_list, 43, 'importance')
    
    all_key_species_shap.update(shap_common_43)
    all_key_species_importance.update(imp_common_43)
    
    print(f'  {group_name}: SHAP={len(shap_common_43)}, Importance={len(imp_common_43)}')

all_key_species_shap = sorted(all_key_species_shap)
all_key_species_importance = sorted(all_key_species_importance)
all_key_species_union = sorted(set(all_key_species_shap) | set(all_key_species_importance))

print(f'\n' + '-'*80)
print(f'Total unique species from SHAP top 43: {len(all_key_species_shap)}')
print(f'Total unique species from Importance top 43: {len(all_key_species_importance)}')
print(f'Combined unique species (union): {len(all_key_species_union)}')
print(f'Using SHAP-only for dependence plots: {len(all_key_species_shap)}')
print('-'*80)

# Save SHAP-only for dependence plots
pd.DataFrame({'Species': all_key_species_shap}).to_excel(
    'results/common_species/key_species_for_plots.xlsx',
    index=False, sheet_name='Key Species')

# Also save union for reference
pd.DataFrame({'Species': all_key_species_union}).to_excel(
    'results/common_species/key_species_union.xlsx',
    index=False, sheet_name='Key Species')

if len(all_key_species_shap) <= 30:
    print('\nAll key SHAP species:')
    for sp in all_key_species_shap:
        print(f'  - {sp}')
else:
    print(f'\nFirst 30 key SHAP species:')
    for sp in all_key_species_shap[:30]:
        print(f'  - {sp}')
    print(f'  ... and {len(all_key_species_shap) - 30} more')

print(f'\nSaved: results/common_species/key_species_for_plots.xlsx (SHAP-only)')
print(f'Saved: results/common_species/key_species_union.xlsx (union, for reference)')

# Print discrepancies between SHAP-only and union
importance_only = sorted(set(all_key_species_union) - set(all_key_species_shap))
shap_only_not_imp = sorted(set(all_key_species_shap) - set(all_key_species_importance))

print(f'\nDiscrepancies:')
print(f'-'*80)
print(f'In Importance top 43 but NOT in SHAP top 43 ({len(importance_only)} species):')
for sp in importance_only:
    print(f'  - {sp}')

print(f'\nIn SHAP top 43 but NOT in Importance top 43 ({len(shap_only_not_imp)} species):')
for sp in shap_only_not_imp:
    print(f'  - {sp}')


Key species identification

--------------------------------------------------------------------------------
  sig0.05: SHAP=3, Importance=4
  sig0.005: SHAP=6, Importance=7
  sig0.005_lean1: SHAP=16, Importance=17
  sig0.005_lean0: SHAP=19, Importance=21
  sig0.001_1000m: SHAP=23, Importance=24
  100m: SHAP=33, Importance=34
  300500m: SHAP=20, Importance=19

--------------------------------------------------------------------------------
Total unique species from SHAP top 43: 62
Total unique species from Importance top 43: 61
Combined unique species (union): 65
Using SHAP-only for dependence plots: 62
--------------------------------------------------------------------------------

First 30 key SHAP species:
  - Alpinia purpurata
  - Ambassis miops
  - Areca catechu
  - Astronia smilacifolia
  - Awaous ocellaris or guamensis
  - Belobranchus belobranchus
  - Belobranchus segura
  - Canis lupus familiaris
  - Cecropia pachystachya
  - Chaetonotidae environmental sample
  - Chilomonas

## 10. SHAP Dependence Plots

Create multi-panel dependence plots showing how species read counts relate to prediction influence across all seven models. Orange points indicate species in the top 43 (5%) for that specific model; blue points indicate species outside the top 43 (5%).

In [22]:
print('\n' + '='*80)
print('SHAP Dependence Plots')
print('='*80)

# Prepare species_counts_clean for dependence plots
sample_key = list(datasets.keys())[0]
species_counts_clean = pd.DataFrame(
    datasets[sample_key]['X_raw'],
    columns=feature_names,
    index=datasets[sample_key]['site_ids']
)

# Clean column names (match preprocessing)
species_counts_clean.columns = (
    species_counts_clean.columns
    .str.replace(r'[\\[\\]<>:]', '_', regex=True)  
    .str.strip()
)

print(f'\nPrepared species_counts_clean: {species_counts_clean.shape[0]} samples × {species_counts_clean.shape[1]} species')


def plot_shap_dependence(target_species, species_counts_clean, models, model_keys, p_values):
    """
    Create multi-panel SHAP dependence plots.
    
    Uses the ORIGINAL models (trained on all 47 samples from Notebook 2).
    These models were NOT modified by the CV loop in Section 5.
    """
    
    # Sort by 'Mean SHAP' before selecting top 43
    highlight_models = []
    for key in model_keys:
        shap_df = pd.read_excel(f'results/shap_analysis/shap_{key}.xlsx')
        # CRITICAL: Must sort first!
        top_43 = shap_df.sort_values(by='Mean SHAP', ascending=False).head(43)['Species'].values
        highlight_models.append(target_species in top_43)
    
    # Create figure with 7 subplots
    fig, axes = plt.subplots(1, 7, figsize=(14, 4), sharey=True)
    
    for i, (ax, key) in enumerate(zip(axes, model_keys)):
        # Get the trained pipeline (trained on all 47 samples)
        pipe = models[key]
        
        # Transform data
        X_log = pipe.named_steps['log_transform'].transform(species_counts_clean)
        X_scaled = pipe.named_steps['scaler'].transform(X_log)
        X_df = pd.DataFrame(
            X_scaled, 
            columns=species_counts_clean.columns,
            index=species_counts_clean.index
        )
        
        # Calculate SHAP values
        explainer = shap.TreeExplainer(pipe.named_steps['clf'])
        shap_vals = explainer.shap_values(X_df)
        
        # Extract SHAP values for class 1 (impacted)
        shap_class_1 = shap_vals[:, :, 1] if len(shap_vals.shape) == 3 else shap_vals[1]
        
        # Get values for target species
        idx = X_df.columns.get_loc(target_species)
        x_vals = X_df.iloc[:, idx]
        y_vals = shap_class_1[:, idx]
        
        # Color based on top 43 status
        color = "darkorange" if highlight_models[i] else "steelblue"
        
        # Scatter plot
        ax.scatter(x_vals, y_vals, alpha=0.6, edgecolor='k', s=12, color=color)
        ax.axhline(0, color='gray', linestyle='--', linewidth=1)
        
        # Title formatting
        buffer, t = key.split('m_t')
        threshold_formatted = '0.' + t[1:]  # "t00625" → "0.0625"
        ax.set_title(f'RFChp_{buffer}m_{threshold_formatted}_5f', fontsize=9)
        
        # Add p-value text
        ax.text(0.02, 0.02, p_values[i], transform=ax.transAxes, 
                fontsize=8, verticalalignment='bottom', horizontalalignment='left')
        
        # Formatting
        ax.grid(True, linestyle='--', alpha=0.3)
        ax.tick_params(axis='both', labelsize=8)
        
        # Only show x-axis label on center plot
        if i != 3:
            ax.set_xticklabels([])
    
    # Set labels and title
    axes[3].set_xlabel("Species Counts (Transformed and Scaled)", fontsize=10)
    axes[0].set_ylabel("SHAP Value", fontsize=10)
    fig.suptitle(
        f"SHAP Dependence of {target_species} Across Optimised Random Forest Classifiers (p < 0.05)", 
        weight='bold', 
        fontsize=13
    )
    
    # Add legend
    orange_patch = mpatches.Patch(color='darkorange', label='Top 5% SHAP Feature')
    fig.legend(handles=[orange_patch], loc='lower right', fontsize=8, frameon=True)
    
    # Adjust layout
    fig.tight_layout()
    fig.subplots_adjust(top=0.82, bottom=0.12)
    
    # Save figure
    safe_name = (target_species
        .replace(' ', '_')
        .replace('.', '_')
        .replace(':', '_')
        .replace('/', '_')
        .replace('(', '').replace(')', ''))
    fig.savefig(
        f'results/shap_analysis/figures/shap_dependence_{safe_name}_multi_model.png', 
        dpi=300, 
        bbox_inches='tight'
    )
    plt.close()


# Model order and p-values
model_keys = [
    '1000m_t00625',      # Position 1
    '1000m_t003125',     # Position 2
    '300m_t003125',      # Position 3
    '500m_t00625',       # Position 4
    '300m_t00625',       # Position 5
    '100m_t00625',       # Position 6
    '100m_t003125'       # Position 7
]

p_values = [
    "p < 0.001",  # 1000m_t00625
    "p < 0.001",  # 1000m_t003125
    "p = 0.004",  # 300m_t003125
    "p = 0.005",  # 500m_t00625
    "p = 0.010",  # 300m_t00625
    "p = 0.024",  # 100m_t00625
    "p = 0.043"   # 100m_t003125
]

# Create plots for all key species
print(f'\nCreating dependence plots for {len(all_key_species_shap)} species...\n')
for i, species in enumerate(all_key_species_shap, 1):
    print(f'  [{i}/{len(all_key_species_shap)}] {species}')
    plot_shap_dependence(species, species_counts_clean, models, model_keys, p_values)

print(f'\nCreated {len(all_key_species_shap)} dependence plots')
print('  Saved to: results/shap_analysis/figures/')


SHAP Dependence Plots

Prepared species_counts_clean: 47 samples × 861 species

Creating dependence plots for 62 species...

  [1/62] Alpinia purpurata
  [2/62] Ambassis miops
  [3/62] Areca catechu
  [4/62] Astronia smilacifolia
  [5/62] Awaous ocellaris or guamensis
  [6/62] Belobranchus belobranchus
  [7/62] Belobranchus segura
  [8/62] Canis lupus familiaris
  [9/62] Cecropia pachystachya
  [10/62] Chaetonotidae environmental sample
  [11/62] Chilomonas sp. M1303
  [12/62] Cochliopodium arabianum
  [13/62] Conium maculatum
  [14/62] Crenimugil cf. heterocheilos
  [15/62] Cyclidium glaucoma
  [16/62] Diplarpea paleacea
  [17/62] Enchelyodon sp. 2 VBOEF-2011
  [18/62] Etoschophrya inornata
  [19/62] Fagraea berteroana
  [20/62] Ficus benguetensis
  [21/62] Ficus prasinicarpa
  [22/62] Gephyramoeba sp. ATCC50654
  [23/62] Gironniera subaequalis
  [24/62] Giuris margaritaceus
  [25/62] Goniomonas truncata
  [26/62] Intsia bijuga
  [27/62] Kleinhovia hospita
  [28/62] Korotnevella jepp

## 11. Summary

**Outputs** (`results/feature_importance/`, `results/shap_analysis/`, `results/common_species/`):

- **Feature importance tables** (`results/feature_importance/`): 7 Excel files (`importance_[model].xlsx`), one per significant model.
- **SHAP visualisations** (`results/shap_analysis/figures/`): Per significant model:
  - `shap_summary_[model].png`
  - `importance_comparison_[model].png`
- **SHAP dependence plots** (`results/shap_analysis/figures/`): `shap_dependence_[species].png`
- **Cross-model species summaries** (`results/common_species/`): 
  - `common_species_summary.xlsx`
  - `key_species_for_plots.xlsx` (final list)


## Next Steps

Analysis complete.